# EmpowerLens - DistilBERT multi-label, wellally.tech tutorial recipe

Runs the recipe from
[wellally.tech/blog/python-cognitive-distortion-transformer-tutorial](https://www.wellally.tech/blog/python-cognitive-distortion-transformer-tutorial)
on the **real** annotated data instead of the tutorial's 15-row toy CSV.

**The recipe, kept verbatim:** `distilbert-base-uncased`,
`problem_type="multi_label_classification"` (plain BCE, no class weighting),
`lr=2e-5`, `batch_size=8`, `weight_decay=0.01`, eval every epoch,
`load_best_model_at_end`, sigmoid with a fixed **0.5** threshold, scored with
micro-F1 / ROC-AUC / accuracy.

**What we changed, and why:**

| Tutorial | Here | Why |
|---|---|---|
| 15 hand-written rows | `Annotated_data.csv` via `data/splits/` (2,024 train / 253 val / 253 test) | 15 rows cannot produce a number that means anything |
| `dataset.train_test_split(test_size=0.2)` each run | the committed frozen splits | project rule: splits are immutable, so runs stay comparable |
| `evaluation_strategy`, `tokenizer=`, `return_all_scores` | `eval_strategy`, `processing_class=`, `top_k=None` | transformers 5.x renamed all three |
| ROC-AUC on 0/1 predictions | ROC-AUC on probabilities (tutorial's version kept alongside) | AUC over binarized labels throws away the ranking it exists to measure |
| "accuracy" as a headline | reported as `subset_accuracy`, F1 is the headline | on a 10-column label matrix, `accuracy_score` = all 10 columns right at once |
| val curve only | val **and train** scored every epoch, plus a data-size sweep | a val curve alone cannot tell underfitting from overfitting - see sections 4b and 4c |
| flat 0.5 everywhere | flat 0.5 on val, **per-class cut points swept on val** applied on test | matches how `experiments/` and the cascade score, so the numbers are comparable; both are reported in 6c |
| `epochs=10` | **`epochs=12`** | matches the cascade's `RECIPE` budget so the two are read on the same axis; `load_best_model_at_end` keeps the best epoch either way, so the extra two cost time, not accuracy |

The 10 label columns are the union of `Dominant Distortion` and
`Secondary Distortion (Optional)`; an all-zero row means *No Distortion*.
`test.csv` is only opened in the last section, by `src/evaluate.py`.

All logic lives in `src/tutorial_distilbert.py` - this notebook only drives it
and displays the results.

## 0. Environment

Skip the next cell when running locally from a clone. On **Kaggle**: Settings ->
Accelerator **GPU**, Internet **On**, then run it.

In [ ]:
# --- Kaggle only: clone the repo and install the transformer stack ---
# REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
# BRANCH   = "nayab-space"
# !rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
# %cd empowerlens
# !pip install -q -r requirements-transformer.txt

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# Work from the repo root no matter where the notebook was launched from.
here = Path.cwd()
if (here / "src" / "tutorial_distilbert.py").exists():
    ROOT = here
elif (here.parent / "src" / "tutorial_distilbert.py").exists():
    ROOT = here.parent          # launched from notebooks/
else:
    raise SystemExit(f"Cannot find src/tutorial_distilbert.py from {here}")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows OpenMP guard
# Pin to one GPU. Kaggle's "T4 x2" accelerator deadlocks this stack across
# devices - every other Kaggle runner in this repo sets this, and omitting it
# is a hang, not an error message.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch, transformers, pandas as pd, numpy as np

# Prefer the project venv's interpreter when this notebook runs on a kernel
# that isn't it; fall back to whatever is running us (Kaggle, Colab, venv).
PY = sys.executable
venv_py = ROOT / "venv" / "Scripts" / "python.exe"
if venv_py.exists() and "venv" not in PY.lower():
    PY = str(venv_py)

CUDA = torch.cuda.is_available()
print(f"python      : {PY}")
print(f"torch       : {torch.__version__}  |  transformers: {transformers.__version__}")
print(f"device      : {'cuda - ' + torch.cuda.get_device_name(0) if CUDA else 'cpu'}")

## 1. The data we are actually training on

Straight from the frozen splits - no reshuffling, no regeneration.

In [ ]:
from src.data import DISTORTIONS
from src.tutorial_distilbert import load_split, get_labels, ML_COLS

splits = {name: load_split("data/splits", name) for name in ("train", "val", "test")}
Y = {name: get_labels(df) for name, df in splits.items()}

manifest = json.loads(Path("data/splits/split_manifest.json").read_text())
print("rows per split:", {k: len(v) for k, v in splits.items()})
print("split random_state:", manifest.get("random_state", "?"), "\n")

dist = pd.DataFrame(
    {name: Y[name].sum(axis=0).astype(int) for name in ("train", "val", "test")},
    index=DISTORTIONS,
)
dist.loc["(no distortion: all-zero row)"] = [
    int((Y[n].sum(axis=1) == 0).sum()) for n in ("train", "val", "test")
]
dist["train_%"] = (100 * dist["train"] / len(splits["train"])).round(1)
display(dist)

lab_per_row = Y["train"].sum(axis=1)
print(f"labels per training row: mean {lab_per_row.mean():.2f}  |  "
      f"0 labels: {(lab_per_row==0).sum()}  1: {(lab_per_row==1).sum()}  "
      f"2: {(lab_per_row==2).sum()}")
print("\nRarest classes drive macro-F1 - watch these in the per-class table later:")
print(dist["train"][DISTORTIONS].sort_values().head(3).to_string())

## 2. Configure the run

The defaults below are the tutorial's hyperparameters **except the epoch
budget**: `EPOCHS = 12`, not the tutorial's 10, to match the cascade's `RECIPE`
in `notebooks/cascade_bootstrap.py`. 12 epochs x 512 tokens is roughly 20
min/seed on a T4 but many hours on CPU, so the CPU path shortens it and says so.

**What matching the epoch count does and does not buy you.** It removes one
axis of difference, which is worth having. It does *not* make this notebook's
numbers interchangeable with the cascade's - that run uses
`mental/mental-roberta-base`, batch size 16 and `--deterministic`, against
DistilBERT at batch size 8 here. And it does not line up with Track A either:
`experiments/kaggle_runner_flat_experiments.ipynb` runs **8** epochs. Quote
across tracks only via the yardstick test set, and say which recipe produced
each row.

Raising 10 to 12 cannot cost accuracy: `load_best_model_at_end` restores the
best epoch by val score, so two extra epochs can only surface a later peak or
be thrown away. Section 4b reports which epoch actually won.

`SEEDS = [42, 1337, 2024]` is the project's three-seed protocol (results are
reported as mean +/- std). Start with one seed to see it work.


In [ ]:
# Swap MODEL to compare backbones. Keep every other setting identical when
# you do, or the comparison confounds architecture with training budget.
#   "distilbert-base-uncased"        the tutorial's
#   "mental/mental-roberta-base"     domain-pretrained, used in results_RUN1/
#   "roberta-base"                   the general-purpose baseline
MODEL      = "distilbert-base-uncased"
TASK       = "multilabel"      # binary | multiclass | multilabel
SEEDS      = [42]              # -> [42, 1337, 2024] for the full protocol
LR         = 2e-5              # tutorial
BATCH_SIZE = 8                 # tutorial
WEIGHT_DEC = 0.01              # tutorial
THRESHOLD  = 0.5               # tutorial
# Which cut points src/evaluate.py applies on TEST in section 6.
#   "tuned"  per-class thresholds swept on VAL (test never sees the sweep).
#            This is what experiments/ and the cascade already do, so it is the
#            setting that makes section 6 comparable with those tables.
#   "fixed"  the tutorial's flat 0.5 - a faithful replication, but then the
#            test number is NOT comparable with the other tracks.
# Either way section 6c reports all four threshold x cap combinations, so
# nothing is hidden by this choice; it only decides which one is the headline
# and which one lands in paper_comparison.csv and the aggregate tables.
META_THRESHOLDS = "tuned"
QUIET      = True              # drop per-step progress bars; keeps epoch metrics
OUT        = "results_RUN2/results_tutorial_distilbert"   # RUN2 root; RUN1 is frozen history


# --- under/overfitting diagnostics (sections 4b and 4c) ---------------------
# TRAIN_EVAL_ROWS: how many TRAINING rows to re-score, in eval mode, at the end
# of every epoch. Overfitting is the GAP between train and val, so without a
# train-side number there is nothing to subtract and the question cannot be
# answered. 512 is twice the val set, so the gap is signal rather than noise,
# and it costs about a fifth of an epoch. 0 turns it off.
TRAIN_EVAL_ROWS = 512

# EARLY_STOP: patience in epochs, 0 = off. Off is the tutorial's behaviour and
# what every number in results_RUN2/ was produced under. Because
# load_best_model_at_end is already on, patience does NOT change the weights
# you keep - it only stops paying for the epochs after the peak.
EARLY_STOP = 0

# FRACS: the data-size axis of section 4c. Each fraction trains a separate
# model on that share of train.csv, with val kept whole.
FRACS = [0.25, 0.5, 1.0]

if CUDA:
    # 12, not the tutorial's 10, to match the cascade's RECIPE in
    # notebooks/cascade_bootstrap.py ("--epochs 12"). Same budget = one fewer
    # axis varying when these numbers are read next to Track B's.
    #
    # It does NOT match Track A: experiments/kaggle_runner_flat_experiments.ipynb
    # runs EPOCHS = 8. Nothing in this repo uses one budget everywhere, so an
    # epoch count alone never makes two runs comparable - see the note below.
    #
    # Raising 10 -> 12 cannot make the result worse: load_best_model_at_end
    # keeps the best epoch either way, so the two extra epochs can only find a
    # later peak or be discarded. They cost time, not accuracy.
    EPOCHS, MAX_LEN = 12, 512
    est = "~20 min per seed on a T4 (12 epochs + the per-epoch train-set pass)"
else:
    EPOCHS, MAX_LEN = 3, 256           # CPU: shortened so it finishes today
    est = "~60-90 min per seed on CPU - use a GPU for the full 12 x 512 run"

print(f"model={MODEL} seeds={SEEDS} epochs={EPOCHS} lr={LR} bs={BATCH_SIZE} "
      f"max_len={MAX_LEN} threshold={THRESHOLD}")
print("estimated:", est)
if not CUDA:
    print("\nNOTE: EPOCHS/MAX_LEN differ from the configured 12/512 because "
          "this is CPU.\n      Numbers below are therefore a lower bound, not "
          "the recipe's ceiling.")
elif EPOCHS != 12:
    print(f"\nNOTE: EPOCHS={EPOCHS} no longer matches the cascade's 12, so "
          f"section 6 is not\n      budget-matched to Track B.")

### Optional: a 2-minute smoke test first

64 rows, 1 epoch. Proves the plumbing runs; the metrics it prints are noise.

In [ ]:
smoke = subprocess.run(
    [PY, "-m", "src.tutorial_distilbert", "--smoke", "--max-length", "128",
     "--out", "results_tutorial_distilbert_smoke", "--no-demo"],
    cwd=ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(smoke.stdout[-1500:] if smoke.returncode == 0
      else smoke.stdout[-3000:] + smoke.stderr[-3000:])
print("\nsmoke test:", "PASSED" if smoke.returncode == 0 else "FAILED")

## 3. Train

Streams the training log live. Each seed trains, picks its best epoch by **val
micro-F1**, then writes metrics, per-class CSVs and a checkpoint.

In [ ]:
cmd = [PY, "-m", "src.tutorial_distilbert",
       "--model", MODEL,
       "--seeds", ",".join(str(s) for s in SEEDS),
       "--epochs", str(EPOCHS),
       "--lr", str(LR),
       "--batch-size", str(BATCH_SIZE),
       "--weight-decay", str(WEIGHT_DEC),
       "--threshold", str(THRESHOLD),
       "--max-length", str(MAX_LEN),
       "--task", TASK,
       "--meta-thresholds", META_THRESHOLDS,
       "--train-eval-rows", str(TRAIN_EVAL_ROWS),
       "--early-stopping-patience", str(EARLY_STOP),
       "--out", OUT] + (["--quiet"] if QUIET else [])
print(" ".join(cmd), "\n")

proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True,
                        encoding="utf-8", errors="replace", bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nexit code:", proc.returncode)

## 4. Results - validation set

Headline is **micro-F1** (what the tutorial optimises). `macro_f1` weights all
ten distortions equally and is the harder, more honest number on this label
distribution.

In [ ]:
out = Path(OUT)
summary  = pd.read_csv(out / f"val_summary_{TASK}.csv", index_col=0)
per_seed = pd.read_csv(out / f"val_metrics_per_seed_{TASK}.csv", index_col=0)

print(f"Validation results - {len(per_seed)} seed(s): {list(per_seed.index)}\n")
display(summary[["mean_pm_std"]] if len(per_seed) > 1 else per_seed.T.round(4))

m = summary["mean"]

# Every metric below the headline is task-shaped. subset_accuracy, hamming_loss
# and labels-per-row only exist for independent sigmoids; binary and multiclass
# are a single softmax, so they have no threshold, no tuned scoring, and
# summarize() writes no val_summary_tuned_* file for them. Reading one
# unconditionally raised FileNotFoundError the moment TASK was switched away
# from multilabel - which the config cell explicitly invites.
if TASK == "multilabel":
    print(f'''
headline    micro-F1        {m["micro_f1"]:.3f}
            macro-F1        {m["macro_f1"]:.3f}   (all 10 classes weighted equally)
            weighted-F1     {m["weighted_f1"]:.3f}
ranking     ROC-AUC micro   {m["roc_auc_micro"]:.3f}   (on probabilities)
            ROC-AUC micro   {m["roc_auc_micro_tutorial"]:.3f}   (tutorial's version, on 0/1 preds)
strictness  subset accuracy {m["subset_accuracy"]:.3f}   (all 10 columns correct at once)
            hamming loss    {m["hamming_loss"]:.3f}
behaviour   labels fired    {m["mean_labels_predicted"]:.2f} per row vs {m["mean_labels_true"]:.2f} actual
''')
    if m["mean_labels_predicted"] < 0.5 * m["mean_labels_true"]:
        print("The model is under-firing: unweighted BCE on rare positives "
              "collapses toward predicting nothing.\nThat is the recipe's main "
              "weakness - see section 6b.")

    # Same probabilities, per-class thresholds swept on val instead of a flat
    # 0.5. This is the val-side "both scorings" view; section 6 applies ONE of
    # them to test (META_THRESHOLDS) and section 6c grids all four on test.
    tuned = pd.read_csv(out / f"val_summary_tuned_{TASK}.csv", index_col=0)["mean"]
    cmp = pd.DataFrame({"@ fixed 0.5": m, "@ tuned": tuned}).loc[
        ["micro_f1", "macro_f1", "roc_auc_micro", "mean_labels_predicted"]]
    cmp["delta"] = cmp["@ tuned"] - cmp["@ fixed 0.5"]
    print("\nThreshold alone, no retraining (ROC-AUC cannot move - it never "
          "used the threshold):")
    display(cmp.round(3))
    print(f"Section 6 will score TEST at: {META_THRESHOLDS}")
else:
    extra = ("positive-class F1", "positive_class_f1") if TASK == "binary" else \
            ("macro-F1 (10, no_distortion dropped)", "macro_f1_10")
    print(f'''
headline    macro-F1        {m["macro_f1"]:.3f}
            weighted-F1     {m["weighted_f1"]:.3f}
            {extra[0]:<36} {m[extra[1]]:.3f}
ranking     ROC-AUC         {m["roc_auc"]:.3f}   (on probabilities)
''')
    print(f"TASK={TASK} predicts with argmax over a single softmax, so there is "
          f"no threshold\nto tune and no fixed-vs-tuned comparison to draw. "
          f"Sections 6c is a no-op too.")


### Per-class breakdown

`support` is the number of val rows carrying that label - a class with 4
positives out of 253 can post F1 = 0.00 from a single miss.

In [ ]:
pc = pd.read_csv(out / f"per_class_val_mean_{TASK}.csv", index_col=0)
display(pc.round(3).sort_values("f1", ascending=False))

dead = pc.index[pc["f1"] == 0].tolist()
if dead:
    print(f"{len(dead)}/10 classes never predicted correctly at threshold "
          f"{THRESHOLD}: {', '.join(dead)}")

png = out / f"per_class_val_f1_{TASK}.png"
try:
    from IPython.display import Image
    if png.exists():
        display(Image(str(png)))
except ImportError:
    print("chart written to", png)

## 4b. Diagnostic 1 - underfitting vs overfitting, epoch by epoch

The old version of this section plotted **val** loss and **val** F1 only. That
shows *when* the val curve peaked, but it cannot tell you *why* it stopped
there, because the two failure modes look identical from the val side:

| what you see on val | underfitting | overfitting |
|---|---|---|
| val F1 stuck low | yes | yes |
| val loss stops falling | yes | yes |
| **train F1** | also low | **high** |
| **train - val gap** | small | **large** |

So the deciding evidence is the *training* score, and until now nothing in this
repo recorded one. `--train-eval-rows` fixes that: after every epoch, a fixed
512-row slice of `train.csv` is scored **the same way val is** - dropout off,
one frozen set of weights, the same metric code. Both series then live in
`epoch_history_*.csv` and can honestly be subtracted.

The Trainer's own per-epoch `loss` is *not* that number. It is a running mean
collected *during* the epoch with dropout on while the weights are still
moving, so it is systematically higher than a clean pass over the same rows. It
is plotted below as a dashed line for reference, but the gap is measured from
`trainset_*`.

**How to read the plot.** Left panel: the two loss curves. They start together;
the epoch where val loss turns back up while train loss keeps falling is the
onset of overfitting. Middle: the same for F1. Right: the gap itself - a rising
line is memorisation. A run where *both* curves sit low and the gap stays flat
near zero is underfitting, and the fix is the opposite one (more capacity,
more epochs, higher LR - not more regularisation).


In [ ]:
import matplotlib.pyplot as plt

# Section 3's runs only. The _frac* files are section 4c's partial-data runs:
# globbing them in here would draw them as if they were extra seeds of the same
# model, and f.stem.split("_")[-1] would label one of them "seed frac50".
hist_files = [f for f in sorted(out.glob(f"epoch_history_{TASK}_*.csv"))
              if "_frac" not in f.stem]
assert hist_files, (f"no full-data epoch_history_{TASK}_*.csv in {out} - "
                    f"run section 3 first")

SEL = {"multilabel": "micro_f1", "binary": "positive_class_f1",
       "multiclass": "macro_f1_10"}[TASK]
EK, TK = f"eval_{SEL}", f"trainset_{SEL}"

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
verdicts = []
for f in hist_files:
    h = pd.read_csv(f)
    seed = f.stem.split("_")[-1]        # tag is <task>_<model>_<seed>
    has_train = TK in h.columns and h[TK].notna().any()

    # -- left: loss ---------------------------------------------------------
    axes[0].plot(h["epoch"], h["eval_loss"], marker="o", label=f"val (seed {seed})")
    if "trainset_loss" in h and h["trainset_loss"].notna().any():
        axes[0].plot(h["epoch"], h["trainset_loss"], marker="s", ls="--",
                     label=f"train (seed {seed})")
    if "train_loss_running" in h and h["train_loss_running"].notna().any():
        axes[0].plot(h["epoch"], h["train_loss_running"], ls=":", alpha=0.5,
                     label=f"train, running (seed {seed})")

    # -- middle: the selection metric --------------------------------------
    axes[1].plot(h["epoch"], h[EK], marker="o", label=f"val (seed {seed})")
    if has_train:
        axes[1].plot(h["epoch"], h[TK], marker="s", ls="--",
                     label=f"train (seed {seed})")

    # -- right: the gap -----------------------------------------------------
    if has_train:
        axes[2].plot(h["epoch"], h[TK] - h[EK], marker="o", label=f"seed {seed}")

    # -- the verdict, in words ---------------------------------------------
    b = int(h[EK].idxmax())
    best_ep, best_val = int(round(h["epoch"].iloc[b])), h[EK].iloc[b]
    row = {"seed": seed, "best_epoch": best_ep, "of": int(round(h["epoch"].max())),
           f"val_{SEL}": round(float(best_val), 3)}
    if has_train:
        tr = float(h[TK].iloc[b])
        row[f"train_{SEL}"] = round(tr, 3)
        row["gap"] = round(tr - float(best_val), 3)
        row["verdict"] = ("UNDERFIT" if tr < 0.55 and tr - best_val < 0.15
                          else "OVERFIT" if tr - best_val > 0.25
                          else "neither, clearly")
    verdicts.append(row)

axes[0].set(xlabel="epoch", ylabel="loss", title="Loss - train vs val")
axes[1].set(xlabel="epoch", ylabel=SEL, title=f"{SEL} - train vs val", ylim=(0, 1))
axes[2].axhline(0, color="k", lw=0.8)
axes[2].set(xlabel="epoch", ylabel=f"train {SEL} - val {SEL}",
            title="The gap (rising = memorising)")
for a in axes:
    a.legend(fontsize=7)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

v = pd.DataFrame(verdicts)
display(v)

if "gap" not in v:
    print("No train-side series in these files. They were written before "
          "--train-eval-rows existed, or TRAIN_EVAL_ROWS was 0.\nRe-run "
          "section 3 to record one.")
else:
    peak_at_end = int((v["best_epoch"] >= v["of"]).sum())
    print(f"mean train-val gap at the best epoch: {v['gap'].mean():+.3f}")
    if peak_at_end:
        print(f"{peak_at_end}/{len(v)} seed(s) peaked at the LAST epoch - the "
              f"curve had not turned over yet, so EPOCHS is too small and "
              f"these numbers are a floor, not a ceiling.")
    else:
        wasted = int((v["of"] - v["best_epoch"]).mean())
        print(f"All seeds peaked before the limit, so {EPOCHS} epochs was "
              f"enough. On average {wasted} epoch(s) were spent past the peak "
              f"- that is what EARLY_STOP would have saved.")
        print("Those epochs cost GPU time but not accuracy: "
              "load_best_model_at_end already threw their weights away.")


## 4c. Diagnostic 2 - is it short of data, or short of capacity?

Section 4b watches one model across epochs. This one asks a different question:
**would more data help?** It trains the same recipe on 25%, 50% and 100% of
`train.csv` and plots the score against the number of training rows. Val is
never subsampled - the exam has to stay the same size, or the curve would be
measuring two things at once.

The shape is the answer:

- **Still climbing at 100%** - the model is *data-limited*. Collecting or
  generating more labelled rows is the highest-value thing you can do, and this
  is the plot that justifies the Month-2 synthetic-data track.
- **Flat / plateaued** - more data will not help. The ceiling is the model, the
  loss, or the label noise. This is the underfitting signature at the dataset
  level, and the fix is a different recipe, not a bigger corpus.
- **Train score high and flat while val stays low, at every size** - classic
  overfitting that more data *would* dilute; the gap should narrow as the
  fractions grow.

`--train-frac` writes each run under its own `_frac25` / `_frac50` suffix, so
these throwaway runs cannot overwrite `val_summary_multilabel.csv` or the epoch
curve from section 3. The 100% point *is* section 3's run, read back off disk -
retraining it would collide with those filenames, and reusing it keeps the
epoch budget identical across the curve.

**Cost:** only the two partial runs, about 0.75x one full training run, on one
seed. On a T4 that is roughly 12 minutes; on CPU, set `RUN_LC = False` and skip
it.


In [ ]:
# The 100% point is NOT retrained here: --train-frac 1.0 writes no suffix, so
# it would land on exactly the filenames section 3 wrote and overwrite that
# run's curve. It is read back off disk instead, which also keeps the epoch
# budget identical across the whole curve - the one thing that must not vary,
# or the plot confounds "less data" with "less training".
#
# CAVEAT, stated rather than hidden: equal EPOCHS at 25% of the data means a
# quarter of the gradient steps. That is the standard convention for a learning
# curve, but it means the left-hand points are mildly pessimistic - part of the
# drop is less optimisation, not less data.
LC_SEED   = SEEDS[0]
LC_EPOCHS = EPOCHS          # identical to section 3 on purpose - see above
RUN_LC    = True            # False = just re-plot whatever is already on disk

for frac in FRACS:
    tag_suffix = "" if frac >= 1.0 else f"_frac{int(round(frac * 100))}"
    done = list(out.glob(f"epoch_history_{TASK}_*_{LC_SEED}{tag_suffix}.csv"))
    if frac >= 1.0:
        print(f"frac 1.0: reusing section 3's run "
              f"({'found' if done else 'NOT FOUND - run section 3 first'})")
        continue
    if done and not RUN_LC:
        print(f"frac {frac}: already on disk, skipping")
        continue
    if not RUN_LC:
        continue
    lc_cmd = [PY, "-m", "src.tutorial_distilbert",
              "--model", MODEL, "--task", TASK,
              "--seeds", str(LC_SEED),
              "--epochs", str(LC_EPOCHS),
              "--lr", str(LR), "--batch-size", str(BATCH_SIZE),
              "--weight-decay", str(WEIGHT_DEC), "--threshold", str(THRESHOLD),
              "--max-length", str(MAX_LEN),
              "--train-frac", str(frac),
              "--train-eval-rows", str(TRAIN_EVAL_ROWS),
              "--early-stopping-patience", str(EARLY_STOP),
              "--no-demo", "--quiet",
              "--out", OUT]
    print(f"\n{'=' * 70}\ntrain-frac {frac}  ({LC_EPOCHS} epochs, seed "
          f"{LC_SEED})\n{'=' * 70}")
    p = subprocess.Popen(lc_cmd, cwd=ROOT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise SystemExit(f"train-frac {frac} failed with exit {p.returncode}")


In [ ]:
# Read the fractions back off disk and plot score vs number of training rows.
N_TRAIN_FULL = len(pd.read_csv("data/splits/train.csv"))

pts = []
for frac in FRACS:
    tag_suffix = "" if frac >= 1.0 else f"_frac{int(round(frac * 100))}"
    matches = sorted(out.glob(f"epoch_history_{TASK}_*_{LC_SEED}{tag_suffix}.csv"))
    if not matches:
        print(f"frac {frac}: no run found, skipping")
        continue
    h = pd.read_csv(matches[-1])
    b = int(h[EK].idxmax())
    row = {"frac": frac, "n_train": int(round(N_TRAIN_FULL * frac)),
           f"val_{SEL}": float(h[EK].iloc[b])}
    if TK in h.columns and h[TK].notna().any():
        row[f"train_{SEL}"] = float(h[TK].iloc[b])
    pts.append(row)

lc = pd.DataFrame(pts).sort_values("n_train")
display(lc.round(3))

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(lc["n_train"], lc[f"val_{SEL}"], marker="o", label=f"val {SEL}")
if f"train_{SEL}" in lc:
    ax.plot(lc["n_train"], lc[f"train_{SEL}"], marker="s", ls="--",
            label=f"train {SEL}")
ax.set(xlabel="training rows", ylabel=SEL, ylim=(0, 1),
       title=f"Learning curve - {SEL} vs training-set size")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

if len(lc) >= 2:
    last_step = float(lc[f"val_{SEL}"].iloc[-1] - lc[f"val_{SEL}"].iloc[-2])
    rows_added = int(lc["n_train"].iloc[-1] - lc["n_train"].iloc[-2])
    print(f"Last {rows_added} training rows moved val {SEL} by {last_step:+.3f}.")
    if last_step > 0.02:
        print("Still climbing at 100% of the data -> DATA-LIMITED. More labelled "
              "rows is the highest-value next move,\nwhich is the argument for "
              "the synthetic-data track.")
    elif last_step < 0.005:
        print("Flat at 100% of the data -> NOT data-limited. The ceiling is the "
              "recipe (model / loss / thresholds)\nor label noise, so more rows "
              "would buy little. See the loss ablation in 6b.")
    else:
        print("Shallow but non-zero slope - more data would help a little. "
              "Weigh it against a recipe change.")
    if f"train_{SEL}" in lc:
        gaps = lc[f"train_{SEL}"] - lc[f"val_{SEL}"]
        print(f"train-val gap by size: "
              + ", ".join(f"{int(n)} rows {g:+.3f}"
                          for n, g in zip(lc['n_train'], gaps)))
        print("A gap that shrinks as the corpus grows is overfitting being "
              "diluted by data.")


## 5. Inference - the tutorial's `predict.py`

The three sentences from the tutorial, plus room for your own.

In [ ]:
from transformers import pipeline
from src.tutorial_distilbert import DEMO_TEXTS

ckpt = Path("checkpoints") / f"tutorial_{TASK}_{MODEL.split('/')[-1]}_{SEEDS[-1]}"
# top_k=None is the transformers 5.x spelling of return_all_scores=True
clf = pipeline("text-classification", model=str(ckpt), tokenizer=str(ckpt),
               top_k=None, device=0 if CUDA else -1)

MY_TEXTS = [
    "If I don't get this right the whole semester is ruined.",
]

from src.tutorial_distilbert import MAX_LABELS_PER_ROW

# Top 2, because the data caps at 2 labels per row (dominant + optional
# secondary) and src/evaluate.py scores test with the same cap. The ranking is
# shown regardless of the threshold: an under-firing model has a useful ranking
# and useless confidence, and only the ranking would otherwise be invisible.
for text in DEMO_TEXTS + MY_TEXTS:
    scores = sorted(clf(text, truncation=True)[0], key=lambda d: d["score"], reverse=True)
    print(f"\n{text!r}")
    for rank, s in enumerate(scores[:MAX_LABELS_PER_ROW], start=1):
        mark = "[fired]" if s["score"] > THRESHOLD else "       "
        print(f"   {mark} {rank}. {s['label']:<22} {s['score']:.4f}")
    if not any(s["score"] > THRESHOLD for s in scores[:MAX_LABELS_PER_ROW]):
        print(f"           nothing crossed {THRESHOLD} - ranking still "
              f"informative, confidence is not")

## 6. Test set

The one place `test.csv` is read, and only through `src/evaluate.py` - the
module the project designates for it. Run this **once**, after you have stopped
changing the recipe; every tuning decision above was made on val.

**Which thresholds get applied here is `META_THRESHOLDS` from section 2.**
`evaluate.py` reads them out of the checkpoint's `meta.json`, so the cell below
first makes that file say what you asked for. Both sets are always saved at
training time (`thresholds_fixed` and `thresholds_tuned`), which means
switching does **not** require retraining - the weights and the probabilities
are identical either way, only the cut point moves.

At `"tuned"` the ten cut points were swept on **val** and are being applied,
unchanged, to test. That is not leakage - test never took part in the sweep -
but it does mean the test number now includes a decision fitted on 253 val
rows, some classes of which have only a handful of positives. The val-to-test
drop printed at the bottom is where that shows up.

`--max-labels 0` means "no cap on how many labels may fire", which is the
tutorial's behaviour (it just thresholds at 0.5).


In [ ]:
# No row in the corpus carries more than 2 labels, and evaluate.py defaults to
# that cap - which is what every other model in results_RUN1/ was scored with.
# The tutorial itself is uncapped (it just thresholds); set 0 to reproduce that.
MAX_LABELS = 2

def apply_threshold_mode(ck: Path, mode: str) -> str:
    """Point meta.json["thresholds"] at the requested set, in place.

    evaluate.py reads that one field and nothing else, so this is the whole
    mechanism. It is done here rather than only at training time so that a
    checkpoint trained before META_THRESHOLDS was flipped does not silently get
    scored under the OLD mode - the failure would be invisible, because both
    modes produce a perfectly plausible-looking number.
    """
    mp = ck / "meta.json"
    meta = json.loads(mp.read_text(encoding="utf-8"))
    if meta.get("task") != "multilabel":
        return "argmax - no threshold to set"
    want = (meta.get("thresholds_tuned") if mode == "tuned"
            else meta.get("thresholds_fixed") or [THRESHOLD] * 10)
    if want is None:
        raise SystemExit(f"{mp} has no thresholds_tuned - it predates the field. "
                         f"Retrain, or set META_THRESHOLDS = 'fixed'.")
    if meta.get("thresholds") != want:
        meta["thresholds"] = want
        mp.write_text(json.dumps(meta, indent=2), encoding="utf-8")
        return f"{mode} (meta.json updated) {want}"
    return f"{mode} (already set) {want}"


for seed in SEEDS:
    ck = Path("checkpoints") / f"tutorial_{TASK}_{MODEL.split('/')[-1]}_{seed}"
    if not ck.exists():
        raise SystemExit(f"{ck} not found - run section 3 first")
    print(f"seed {seed}: thresholds -> {apply_threshold_mode(ck, META_THRESHOLDS)}")
    # --reference appends the Shreevastava & Foltz (2021) paper rows to
    # paper_comparison.csv - binary weighted-F1 0.79, multiclass 0.30 - so our
    # numbers sit next to the paper this project replicates.
    cmd = [PY, "-m", "src.evaluate", "--checkpoint", str(ck), "--out", OUT,
           "--reference"]
    if TASK == "multilabel":          # the flag is multilabel-only
        cmd += ["--max-labels", str(MAX_LABELS)]
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    print(r.stdout.strip()[-600:] or r.stderr[-1500:])

rows = []
for f in sorted(Path(OUT).glob(f"eval_*{TASK}*.json")):
    d = json.loads(f.read_text())          # nested: splits -> val/test -> metrics
    for split_name, block in d["splits"].items():
        rows.append({"seed": d["meta"]["seed"], "split": split_name,
                     **{k: v for k, v in block["metrics"].items()
                        if k in ("macro_f1", "micro_f1", "weighted_f1",
                                 "macro_f1_10", "positive_class_f1")
                        and v != ""}})
test_tbl = pd.DataFrame(rows)
display(test_tbl.round(3))

if not test_tbl.empty and {"val", "test"} <= set(test_tbl["split"]):
    key = {"binary": "positive_class_f1",
           "multiclass": "macro_f1_10"}.get(TASK, "macro_f1")
    v = test_tbl[test_tbl.split == "val"][key].mean()
    t = test_tbl[test_tbl.split == "test"][key].mean()
    print(f"\nval {key} {v:.3f} -> test {t:.3f}  (drop of {t - v:+.3f})")
    print(f"thresholds applied: {META_THRESHOLDS}"
          + ("  (swept on val, applied unchanged to test)"
             if META_THRESHOLDS == "tuned" else f"  (flat {THRESHOLD})"))
    print("A drop is expected and healthy: val is the set we picked the epoch "
          "and\nthresholds on, so it flatters. Test is the honest number. A "
          "LARGE drop\nmeans those choices fitted val's noise rather than the "
          "task.")
    if META_THRESHOLDS == "tuned":
        print("Under 'tuned' the drop also absorbs threshold overfitting: ten "
              "cut points\nchosen on 253 val rows. Section 6c's grid shows what "
              "the flat 0.5 would have\nscored on the same weights, which is "
              "the honest way to price that in.")

### Confusion matrices

`src.evaluate` writes these for **binary** and **multiclass**. There is no
confusion matrix for **multilabel** - a confusion matrix needs one predicted
class per row to cross-tabulate against the true one, and multilabel gives each
row up to 10 independent yes/no answers. The per-class precision/recall/F1 table
is the equivalent view for that task.

In [ ]:
pngs = sorted(Path(OUT).glob(f"confusion_*{TASK}*.png"))
if not pngs:
    if TASK == "multilabel":
        print("No confusion matrix for multilabel - by design, see above. "
              "Use the per-class table in Section 4 instead.")
    else:
        print(f"No confusion PNGs in {OUT}/ yet - run the test cell above first.")
else:
    from IPython.display import Image, display as _d
    for p in pngs:
        # "_no_nd" = the same matrix with no_distortion dropped, so the ten
        # distortions are readable without the majority class dominating.
        print(f"\n{p.name}")
        _d(Image(str(p)))

## 6b. Loss ablation - isolating the two fixes

The tutorial's weakness is that ~95% of the gradient on a rare class says "no",
so probabilities never reach 0.5. There are two independent remedies, and this
section measures them separately:

- **Change the loss** - alters what the model *learns*.
- **Change the threshold** - alters only where the decision line sits on the
  *same* probabilities. No retraining; it is a post-hoc rescoring.

Four losses, all sigmoid-based, all scoring the 10 labels independently:

| `--loss` | mechanism |
|---|---|
| `bce` | unweighted `BCEWithLogitsLoss` - the tutorial's |
| `pos_bce` | + `pos_weight = negatives/positives` per class (19.0x for `all_or_nothing`) |
| `focal` | `FocalLoss(gamma=2)` on top of `pos_weight` - also damps *easy* examples |
| `asl` | `AsymmetricLoss` - separate gammas for positives/negatives, replaces `pos_weight` |

Each is trained once and scored at both threshold settings, so this is 4
trainings for 8 result rows. Budget ~4x the section-3 time.

In [ ]:
abl_cmd = [PY, "-m", "src.tutorial_distilbert",
           "--model", MODEL,
           "--seeds", ",".join(str(s) for s in SEEDS),
           "--epochs", str(EPOCHS), "--lr", str(LR),
           "--batch-size", str(BATCH_SIZE), "--weight-decay", str(WEIGHT_DEC),
           "--threshold", str(THRESHOLD), "--max-length", str(MAX_LEN),
           "--task", TASK, "--out", OUT, "--ablation", "--no-demo"]
if QUIET:
    abl_cmd.append("--quiet")
print(" ".join(abl_cmd), "\n")

proc = subprocess.Popen(abl_cmd, cwd=ROOT, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True,
                        encoding="utf-8", errors="replace", bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nexit code:", proc.returncode)

In [ ]:
abl = pd.read_csv(Path(OUT) / "ablation_summary.csv")
abl = abl[abl["task"] == TASK]
display(abl.round(3))

pivot = abl.pivot(index="loss", columns="threshold_mode", values="macro_f1")
pivot = pivot.reindex([l for l in ["bce", "pos_bce", "focal", "asl",
                                   "ce", "weighted_ce"] if l in pivot.index])
ax = pivot.plot.bar(figsize=(8, 4), rot=0,
                    color={"fixed 0.5": "#C44E52", "tuned": "#4C72B0"})
ax.set_ylabel("val macro-F1")
ax.set_title("Loss x threshold: what each fix is worth on its own")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="threshold")
plt.tight_layout()
plt.show()

base = pivot.loc["bce", "fixed 0.5"]
print(f"tutorial baseline (bce @ 0.5)     : macro-F1 {base:.3f}")
print(f"threshold alone (bce @ tuned)     : {pivot.loc['bce', 'tuned']:.3f}  "
      f"({pivot.loc['bce', 'tuned'] - base:+.3f})")
print(f"loss alone (best loss @ 0.5)      : {pivot['fixed 0.5'].max():.3f}  "
      f"({pivot['fixed 0.5'].max() - base:+.3f})  [{pivot['fixed 0.5'].idxmax()}]")
print(f"both (best loss @ tuned)          : {pivot['tuned'].max():.3f}  "
      f"({pivot['tuned'].max() - base:+.3f})  [{pivot['tuned'].idxmax()}]")
if "mean_labels_predicted" in abl:
    print("\nlabels fired per row (true mean is ~0.79) - this is what separates "
          "under-firing\nfrom spraying, which macro-F1 alone cannot:")
    print(abl.pivot(index="loss", columns="threshold_mode",
                    values="mean_labels_predicted").round(2).to_string())

pr = [c for c in ("macro_precision", "macro_recall", "macro_f1") if c in abl]
if len(pr) == 3:
    print("\nprecision vs recall - F1 is their harmonic mean, so it hides which "
          "way a\nconfiguration is failing:")
    print(abl.set_index(["loss", "threshold_mode"])[pr].round(3).to_string())

### Threshold x cap grid

Two decisions sit between the probabilities and the score, and they interact:

- **threshold** - the tutorial's flat 0.5, or the per-label cut points swept on val
- **cap** - `max_labels=2` (the corpus never has more than 2 labels per row, and
  it is how every model in `results/` was scored) or uncapped (the tutorial's)

Once **two or more** labels clear the threshold, the cap keeps the top 2 by
probability - and the top 2 are the same wherever the line sits. So under a cap,
moving the threshold can change *nothing at all*. Scoring only one combination
would mislead in either direction, so all four are reported.

No retraining: the weights are identical in every pass. `meta.json` is patched
and restored.

In [ ]:
if TASK == "multilabel":
    for seed in SEEDS:
        ck = Path("checkpoints") / f"tutorial_{TASK}_{MODEL.split('/')[-1]}_{seed}"
        if ck.exists():
            run = subprocess.run([PY, "-m", "src.eval_thresholds",
                                  "--checkpoint", str(ck), "--out", OUT],
                                 cwd=ROOT, capture_output=True, text=True,
                                 encoding="utf-8", errors="replace")
            print(run.stdout[-1200:] or run.stderr[-1200:])

    grid = Path(OUT) / "threshold_grid.csv"
    if grid.exists():
        g = pd.read_csv(grid)
        g = g[g["split"] == "test"]
        display(g.pivot_table(index=["threshold_mode", "max_labels"],
                              values=["macro_f1", "micro_f1", "weighted_f1"],
                              aggfunc="mean").round(4))
        print("Quote the max_labels=2 row alongside the other models in results/.")
        print("The uncapped row is the tutorial's own behaviour - label it as such.")
else:
    print(f"TASK={TASK} uses argmax, so there is no threshold to vary.")

## 7. What this tells you

Two things worth writing down after the run:

1. **Where the recipe breaks.** Unweighted BCE plus a flat 0.5 threshold, on a
   corpus where the commonest distortion covers 11.5% of training rows and the
   rarest 5.0%, pushes every sigmoid below 0.5 for the rare classes. Low
   `macro_f1` next to a much healthier `roc_auc_micro` is the signature: the
   model *ranks* correctly but never crosses the threshold. Section 6b
   quantifies which of the two fixes is worth more.

2. **Which fix matters.** If `bce @ tuned` alone recovers most of the gap, the
   model was fine and only the decision line was wrong. If it takes `pos_bce`
   or `asl` to move `macro_f1`, the unweighted loss really did stop it learning
   the rare classes. Those are different findings and worth stating separately
   in a write-up.

3. **The comparison.** The cell below puts this run next to the project's own
   multi-label runs, if you have any in `results/`. Note `roberta-base` is a
   bigger backbone than `distilbert-base`, so that row mixes architecture with
   recipe - section 6b is the controlled comparison, since it holds the
   architecture fixed and varies only the loss.

In [ ]:
mine = pd.DataFrame([{
    "model": f"{MODEL} (tutorial recipe)", "split": "val", "seeds": len(per_seed),
    "micro_f1": summary.loc["micro_f1", "mean"],
    "macro_f1": summary.loc["macro_f1", "mean"],
    "loss": "BCE, unweighted", "threshold": f"fixed {THRESHOLD}",
}])

others = pd.DataFrame()
pcsv = Path("results/paper_comparison.csv")
if pcsv.exists():
    ref = pd.read_csv(pcsv)
    ref = ref[(ref.task == "multilabel") & (ref.split == "val")]
    if not ref.empty:
        others = (ref.groupby("model")
                     .agg(seeds=("seed", "nunique"),
                          micro_f1=("micro_f1", "mean"),
                          macro_f1=("macro_f1", "mean"))
                     .reset_index())
        others["model"] += " (project pipeline)"
        others["split"] = "val"
        others["loss"] = "BCE + pos_weight"
        others["threshold"] = "swept on val"

display(pd.concat([mine, others], ignore_index=True)
          [["model", "split", "seeds", "micro_f1", "macro_f1", "loss", "threshold"]]
          .round(3))

## 7b. Month-1 summary tables

`src.aggregate` is what every earlier runner in this repo calls, and it produces
two things the per-run files do not:

- **`month1_summary_meanstd.csv`** - the headline metrics as mean +/- std across
  seeds, in the same shape as the existing Month-1 tables.
- **`no_distortion_contribution.md`** - for each 11-class run, how much of the
  weighted-F1 comes from the single easy `no_distortion` class. That number is
  the difference between an honest headline and a flattering one, so it is worth
  generating even when the task is multilabel.

In [ ]:
r = subprocess.run([PY, "-m", "src.aggregate", "--results", OUT],
                   cwd=ROOT, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(r.stdout[-1500:] or r.stderr[-1500:])

ms = Path(OUT) / "month1_summary_meanstd.csv"
if ms.exists():
    display(pd.read_csv(ms).round(3))

nd = Path(OUT) / "no_distortion_contribution.md"
if nd.exists():
    print()
    print(nd.read_text(encoding="utf-8")[:1500])

# paper_comparison.csv now carries the literature rows too (source == "paper").
pc = Path(OUT) / "paper_comparison.csv"
if pc.exists():
    d = pd.read_csv(pc)
    display(d[["model", "task", "seed", "split", "weighted_f1", "macro_f1",
               "micro_f1", "source"]].round(3))
    if (d["source"] == "paper").any():
        print()
        print("Rows with source='paper' are Shreevastava & Foltz (2021), "
              "not our runs.")
        print("Their binary 0.79 has UNSPECIFIED averaging - do not quote it "
              "as weighted-F1 without that caveat.")

## 8. Collect the results

Builds `docs/RERUN_EXPERIMENTS.md` from whatever has been run, then copies
everything into `/kaggle/working/` so it appears in the session's **Output**
pane and can be downloaded.

Checkpoints are deliberately NOT copied — they are ~250 MB each and the repo
gitignores them. Only re-run the training if you need the weights again.

In [ ]:
# Roll every run in OUT into one document, grouped by task.
r = subprocess.run([PY, "-m", "src.make_rerun_table", "--results", OUT,
                    "--out", "docs/RERUN_EXPERIMENTS.md"],
                   cwd=ROOT, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(r.stdout or r.stderr)

import shutil
KAGGLE_OUT = Path("/kaggle/working")
if KAGGLE_OUT.exists():
    dest = KAGGLE_OUT / OUT
    shutil.copytree(Path(OUT), dest, dirs_exist_ok=True)
    doc = Path("docs/RERUN_EXPERIMENTS.md")
    if doc.exists():
        shutil.copy(doc, KAGGLE_OUT / doc.name)
    print(f"\ncopied to {KAGGLE_OUT} - check the Output pane:")
    for f in sorted(dest.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(KAGGLE_OUT)}  ({f.stat().st_size:,} b)")
else:
    print(f"\nNot on Kaggle - results are already in {ROOT / OUT}")

# The generated doc, inline.
doc = Path("docs/RERUN_EXPERIMENTS.md")
if doc.exists():
    try:
        from IPython.display import Markdown
        display(Markdown(doc.read_text(encoding="utf-8")))
    except ImportError:
        print(doc.read_text(encoding="utf-8")[:4000])

### Zip it into a single download

One file is easier to pull off Kaggle than thirty. The archive lands in
`/kaggle/working/` and shows up in the **Output** pane on the right - click it
to download.

Weights are excluded on purpose: a DistilBERT checkpoint is ~250 MB each, and
with three tasks x four losses x three seeds that is tens of gigabytes. Set
`INCLUDE_CHECKPOINTS = True` only if you specifically need the weights back,
and expect a slow zip.

In [ ]:
import shutil, zipfile, datetime

INCLUDE_CHECKPOINTS = False        # ~250 MB per run - leave off unless needed

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
dest_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else ROOT
zip_path = dest_dir / f"empowerlens_{TASK}_{stamp}.zip"

def add(zf, path, arc_root):
    # Add one file, or a whole tree, under arc_root inside the archive.
    path = Path(path)
    if path.is_file():
        zf.write(path, Path(arc_root) / path.name)
    elif path.is_dir():
        for f in sorted(path.rglob("*")):
            if f.is_file():
                zf.write(f, Path(arc_root) / f.relative_to(path))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    add(zf, OUT, OUT)                                   # metrics, per-class, plots
    add(zf, "docs/RERUN_EXPERIMENTS.md", "docs")        # the generated write-up
    add(zf, "data/splits/split_manifest.json", "data/splits")   # provenance
    if INCLUDE_CHECKPOINTS:
        for ck in sorted(Path("checkpoints").glob(f"tutorial_{TASK}_*")):
            add(zf, ck, f"checkpoints/{ck.name}")

size_mb = zip_path.stat().st_size / 1e6
with zipfile.ZipFile(zip_path) as zf:
    n = len(zf.namelist())
print(f"{zip_path}  ({size_mb:.1f} MB, {n} files)")
if dest_dir.name == "working":
    print("Find it in the Output pane on the right of the Kaggle editor.")

# The split manifest rides along so a downloaded result can always be traced
# back to the exact data that produced it - checksum, row counts, random_state.

Artifacts written by this notebook:

```
results_RUN2/results_tutorial_distilbert/
  val_summary.csv            mean +/- std over seeds
  val_metrics_per_seed.csv   one row per seed
  val_metrics_<model>_<seed>.json
  per_class_val_mean.csv     precision/recall/F1/support per distortion
  per_class_val_f1.png
  epoch_history_<task>_<model>_<seed>.csv
        one row per epoch. eval_* = val, trainset_* = the 512-row train
        slice scored the same way, train_loss_running = the Trainer's own
        in-epoch mean. The train-vs-val gap in section 4b comes from here.
  *_frac25 / *_frac50 variants   the section 4c learning-curve runs
  demo_predictions.csv
checkpoints/tutorial_<model>_<seed>/   weights + meta.json (gitignored)
```